# PNR Polarisation: 2-Layer NF Inference

Test the 2-layer normalizing-flow checkpoint on `00058_du.dat` and `00058_uu.dat`.

The attached fit notebook models air/Pt/Co/Al2O3. Here Pt and Co are the two inferred layers, and each polarisation channel is tested independently with its effective Co SLD. This is a model smoke test, not a joint magnetic PNR fit.

The workflow loads both polarisation curves, applies channel-specific priors, runs the same L2 checkpoint independently, and compares each MAP result with the copied fit reference.

## 1. Setup

Locate the project root and configure Matplotlib for inline output. The shared helpers keep tensor conversion and device selection consistent across both channels.

In [ ]:
%matplotlib inline

from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib as mpl

mpl.use("module://matplotlib_inline.backend_inline", force=True)

import matplotlib.pyplot as plt
import numpy as np
import torch

# Locate the repository root from either the project root or notebooks/.
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "exp_data" / "00058_du.dat").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the inverse-eval repo.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from device_utils import detect_torch_device, summarize_torch_backends
from nf_statistics import compute_nf_sample_statistics
from reflectorch import EasyInferenceModel


# Convert model outputs to NumPy before statistics and plotting.
def to_numpy(value):
    if torch.is_tensor(value):
        return value.detach().cpu().numpy()
    return np.asarray(value)


plt.rcParams["text.usetex"] = False
torch.manual_seed(42)

print(f"Project root    : {PROJECT_ROOT}")
print(f"PyTorch version : {torch.__version__}")
print(f"Backends        : {summarize_torch_backends()}")

## 2. Configure Channels and Priors

The values copied from the fit notebook define narrow SLD priors and the reference overlay. They are a fit reference for comparison, not independent ground truth.

In [ ]:
# The channels are inferred independently; this checkpoint has no joint magnetic coupling.
DATA_FILES = {
    "du": PROJECT_ROOT / "exp_data" / "00058_du.dat",
    "uu": PROJECT_ROOT / "exp_data" / "00058_uu.dat",
}
FIT_NOTEBOOK = PROJECT_ROOT / "exp_data" / "NR_XRR_Fit_AlO3_Co_Pt.ipynb"
VENDOR_ROOT = PROJECT_ROOT / "vendor" / "nflows_reflectorch"

NF_CONFIG = "example_nf_config_reflectorch_L2.yaml"
NF_SAMPLES = 1000
DEVICE = detect_torch_device()
ENABLE_ERROR_BARS_FILTERING = False  # False matches the NR_XRR fit notebook's full positive-Q NR curves.
FILTER_THRESHOLD = 0.3
FILTER_CONSECUTIVE = 3

# Values read from NR_XRR_Fit_AlO3_Co_Pt.ipynb, in 10^-6 A^-2.
PT_SLD = 6.24996
CO_NUCLEAR_SLD = 2.26454
CO_MAG_SLD_BOUNDS = (3.331345, 4.331345)
AL2O3_SLD = 5.71528
FIXED_SLD_HALF_WIDTH = 0.005

PARAM_ORDER = [
    "Pt thickness",
    "Co thickness",
    "air/Pt roughness",
    "Pt/Co roughness",
    "Co/Al2O3 roughness",
    "Pt SLD",
    "effective Co SLD",
    "Al2O3 SLD",
    "r_scale",
    "log10_background",
]

# Copied from the joint NR/XRR fit notebook for reporting and the SLD overlay only.
FIT_REFERENCE = {
    "du": {
        "Pt thickness": 43.98,
        "Co thickness": 95.05,
        "air/Pt roughness": 8.12,
        "Pt/Co roughness": 6.83,
        "Co/Al2O3 roughness": 4.46,
        "Pt SLD": 6.25,
        "effective Co SLD": -1.75,
        "Al2O3 SLD": 5.72,
    },
    "uu": {
        "Pt thickness": 43.98,
        "Co thickness": 95.05,
        "air/Pt roughness": 8.12,
        "Pt/Co roughness": 6.83,
        "Co/Al2O3 roughness": 4.46,
        "Pt SLD": 6.25,
        "effective Co SLD": 6.28,
        "Al2O3 SLD": 5.72,
    },
}


def fixed_sld_bounds(center):
    return (center - FIXED_SLD_HALF_WIDTH, center + FIXED_SLD_HALF_WIDTH)


def co_sld_bounds(polarisation):
    sign = {"du": -1.0, "uu": 1.0}[polarisation]
    values = [CO_NUCLEAR_SLD + sign * value for value in CO_MAG_SLD_BOUNDS]
    return (min(values), max(values))


def prior_bounds_for(polarisation):
    structural_bounds = [
        (10.0, 100.0),
        (50.0, 250.0),
        (0.0, 10.0),
        (0.0, 10.0),
        (0.0, 10.0),
        fixed_sld_bounds(PT_SLD),
        co_sld_bounds(polarisation),
        fixed_sld_bounds(AL2O3_SLD),
    ]
    return [*structural_bounds, (0.9, 1.1), (-10.0, -4.0)]


print(f"Fit notebook : {FIT_NOTEBOOK}")
print(f"NF config    : {NF_CONFIG}")
print(f"Device       : {DEVICE}")
print(f"Samples      : {NF_SAMPLES}")
for name in DATA_FILES:
    print(f"{name} prior dim : {len(prior_bounds_for(name))}")

### Teaching question 1: same sample, different channels

**Predict / Explain:** Both polarisation channels measure the same sample. Which structural quantities should they share, and which can differ? Does running the same neural network separately on both channels enforce those shared quantities?

<details>
<summary>Instructor answer — click to expand</summary>

The sample has shared layer thicknesses and interface roughnesses. Its effective Co SLD can differ between channels because of the magnetic contribution. Using the same network does not enforce shared geometry: each inference uses its own curve and priors. Different thickness estimates therefore do not imply that the physical sample changed.

</details>

After inference, revisit your prediction using the existing parameter summaries or the optional thickness-comparison snippet below.


## 3. Load Polarisation Curves

Each data file provides Q, reflectivity, uncertainty, and Q resolution. Rows with invalid or non-positive reflectivity are excluded; the retained curves are plotted before inference.

In [ ]:
def load_curve(path):
    data = np.loadtxt(path)
    if data.ndim != 2 or data.shape[1] < 4:
        raise ValueError(f"Expected four columns Q, R, dR, dQ in {path}")

    q, r, dr, dq = data[:, 0], data[:, 1], data[:, 2], data[:, 3]
    # Keep only rows that can be used by log-scale reflectivity inference.
    mask = (
        np.isfinite(q)
        & np.isfinite(r)
        & np.isfinite(dr)
        & np.isfinite(dq)
        & (q > 0)
        & (r > 0)
        & (dr > 0)
        & (dq >= 0)
    )
    return {
        "q": q[mask],
        "r": r[mask],
        "dr": dr[mask],
        "dq": dq[mask],
        "raw_points": len(data),
        "removed_points": int((~mask).sum()),
    }


curves = {name: load_curve(path) for name, path in DATA_FILES.items()}

for name, curve in curves.items():
    print(f"{name}: {len(curve['q'])}/{curve['raw_points']} points kept")
    print(f"  Q range    : {curve['q'].min():.5f} - {curve['q'].max():.5f} A^-1")
    print(f"  R range    : {curve['r'].min():.3e} - {curve['r'].max():.3e}")
    print(f"  dR/R range : {(curve['dr'] / curve['r']).min():.3f} - {(curve['dr'] / curve['r']).max():.3f}")
    print(f"  dQ/Q med.  : {np.median(curve['dq'] / curve['q']):.4f}")
    print(f"  removed    : {curve['removed_points']}")

fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
for name, curve in curves.items():
    ax.errorbar(curve["q"], curve["r"], yerr=curve["dr"], fmt=".", ms=3, alpha=0.65, label=name)
ax.set(xlabel="Q (A^-1)", ylabel="Reflectivity", yscale="log")
ax.legend(title="polarisation")
plt.show()

## 4. Load the L2 Model

Load the vendored checkpoint once, then confirm that each channel's prior vector has the ten dimensions required by the model.

In [ ]:
model = EasyInferenceModel(
    config_name=NF_CONFIG,
    root_dir=str(VENDOR_ROOT),
    repo_id=None,
    device=DEVICE,
)

expected_dim = model.trainer.loader.prior_sampler.param_dim
for name in curves:
    assert len(prior_bounds_for(name)) == expected_dim, (name, len(prior_bounds_for(name)), expected_dim)

print(f"Model prior dim: {expected_dim}")

## 5. Run Independent Channel Inference

`preprocess_and_sample` produces posterior parameters, curves, SLD profiles, and log-likelihoods for each channel. Error-bar filtering remains disabled so the full positive-Q NR range is retained.

In [ ]:
def run_channel(name, curve):
    prior_bounds = prior_bounds_for(name)
    prediction = model.preprocess_and_sample(
        reflectivity_curve=curve["r"],
        q_values=curve["q"],
        sigmas=curve["dr"],
        q_resolution=curve["dq"],
        prior_bounds=prior_bounds,
        num_samples=NF_SAMPLES,
        sampling_batch_size=128,
        maximum_sim_batch_size=128,
        calc_sampled_curves=True,
        calc_sampled_sld_profiles=True,
        calc_log_likelihoods=True,
        enable_importance_sampling=True,
        clip_prediction=True,
        # Preserve the full retained curve instead of applying the default high-error tail filter.
        enable_error_bars_filtering=ENABLE_ERROR_BARS_FILTERING,
        filter_threshold=FILTER_THRESHOLD,
        filter_consecutive=FILTER_CONSECUTIVE,
    )
    return {"prediction": prediction, "prior_bounds": prior_bounds}


results = {name: run_channel(name, curve) for name, curve in curves.items()}

for name, result in results.items():
    prediction = result["prediction"]
    print(f"{name}: params {to_numpy(prediction['predicted_params_array']).shape}, curves {to_numpy(prediction['sampled_curves']).shape}")

## 6. Inspect MAP Estimates

For each channel, the maximum-log-likelihood sample is reported beside posterior summary statistics. The weighted MSE is calculated on the interpolated model grid, while the copied fit values provide a diagnostic comparison.

In [ ]:
def summarize_result(name, result):
    prediction = result["prediction"]
    prior_bounds = result["prior_bounds"]
    log_likelihoods = to_numpy(prediction["log_likelihoods"])
    samples = to_numpy(prediction["predicted_params_array"])
    best_idx = int(np.argmax(log_likelihoods))
    map_params = samples[best_idx]
    stats = compute_nf_sample_statistics(samples, log_likelihoods)

    # Compare the MAP curve on the model's interpolated Q grid.
    interp_r = to_numpy(prediction["reflectivity_curve_interp"])
    interp_dr = to_numpy(prediction.get("sigmas_interp", interp_r * 0.2 + 1e-12))
    map_curve = to_numpy(prediction["sampled_curves"])[best_idx]
    weighted_mse = float(np.mean(((interp_r - map_curve) / interp_dr) ** 2))

    result.update(best_idx=best_idx, map_params=map_params, weighted_mse=weighted_mse)

    print(f"\n{name} MAP index: {best_idx}")
    print(f"{name} MAP loglike: {log_likelihoods[best_idx]:.4f}")
    print(f"{name} weighted MSE: {weighted_mse:.4g}")
    if "ess" in prediction:
        print(f"{name} ESS: {prediction['ess']:.1f} / {NF_SAMPLES}")
    print(f"{'Parameter':<22} {'Prior min':>11} {'MAP':>12} {'Mean':>12} {'Std':>12} {'Prior max':>11}")
    print("-" * 86)
    for i, label in enumerate(PARAM_ORDER):
        lo, hi = prior_bounds[i]
        print(
            f"{label:<22} {lo:>11.4f} {map_params[i]:>12.4f} "
            f"{stats['nf_params_mean'][i]:>12.4f} {stats['nf_params_std'][i]:>12.4f} {hi:>11.4f}"
        )

    print(f"\n{name} MAP vs NR_XRR fit notebook:")
    print(f"{'Parameter':<22} {'NF MAP':>12} {'NR_XRR fit':>12} {'Delta':>12}")
    print("-" * 62)
    for i, label in enumerate(PARAM_ORDER[:8]):
        fit_value = FIT_REFERENCE[name][label]
        print(f"{label:<22} {map_params[i]:>12.4f} {fit_value:>12.4f} {map_params[i] - fit_value:>12.4f}")


for name, result in results.items():
    summarize_result(name, result)

### Optional code snippet: compare selected thickness estimates

**Optional — code suggestion only.** Copy the following into a new code cell and run it after the parameter summaries. It uses the existing results and requires no additional inference. The existing code cells are unchanged.

**Explain:** Do the two independently selected structures give similar Pt and Co thicknesses? How do they compare with the joint-fit reference?

```python
x = np.arange(2)
fig, ax = plt.subplots(figsize=(5, 4))

for offset, name in zip((-0.12, 0.12), ("du", "uu")):
    ax.scatter(x + offset, results[name]["map_params"][:2],
               label=name)

reference = [FIT_REFERENCE["du"][f"{m} thickness"]
             for m in ("Pt", "Co")]
ax.scatter(x, reference, marker="x", color="black",
           label="Joint-fit reference")

ax.set(xticks=x, xticklabels=["Pt", "Co"],
       ylabel="Thickness (Å)", title="Selected thickness estimates")
ax.legend()
plt.show()
```


## 7. Plot MAP Fits and SLD Profiles

Each row shows a channel's measured reflectivity with its MAP curve and the corresponding MAP SLD profile. The two channels are displayed together for inspection, but are not jointly fitted.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)

for row, name in enumerate(["du", "uu"]):
    curve = curves[name]
    result = results[name]
    prediction = result["prediction"]
    best_idx = result["best_idx"]

    q_plot = to_numpy(prediction["q_plot_pred"])
    map_curve = to_numpy(prediction["sampled_curves"])[best_idx]
    sld_x = to_numpy(prediction["sampled_sld_xaxis"])
    map_sld = to_numpy(prediction["sampled_sld_profiles"])[best_idx]

    ax_r, ax_sld = axes[row]
    ax_r.errorbar(curve["q"], curve["r"], yerr=curve["dr"], fmt=".", ms=3, alpha=0.55, label=f"{name} data")
    ax_r.plot(q_plot, map_curve, lw=2, label="NF MAP")
    ax_r.set(xlabel="Q (A^-1)", ylabel="Reflectivity", yscale="log", title=f"{name} reflectivity")
    ax_r.legend()

    ax_sld.plot(sld_x, map_sld, lw=2)
    ax_sld.set(xlabel="Depth (A)", ylabel="SLD (1e-6 A^-2)", title=f"{name} MAP SLD")

plt.show()

### Teaching question 2: are we comparing the same inference problem?

**Explain:** The reference analysis jointly fits X-ray reflectivity and both neutron channels. This notebook treats the neutron channels separately. If their results differ, can we conclude that one inference method is more accurate?

<details>
<summary>Instructor answer — click to expand</summary>

No. They use different information and constraints. The joint analysis combines three curves and couples geometry and magnetism; the NF runs treat each neutron channel independently. The reference values are another fit result, and some reference information also informs the NF priors. The comparison is useful, but does not isolate the accuracy of ML versus conventional fitting.

</details>


## Interpretation note

The two NF runs infer each polarisation channel independently; they do not enforce shared geometry or magnetic coupling. The reference analysis jointly fits X-ray reflectivity and both neutron channels, and some reference values also informed these priors. Differences between the results therefore do not isolate the accuracy of either method.

Error-bar filtering is disabled so inference uses the full set of retained positive-reflectivity rows. Distinct channel results can indicate model mismatch or different information in the curves; they do not mean the sample geometry changed.


## 8. NF Sampled SLD Profiles

Overlay every finite SLD profile sampled by the NF posterior for each polarisation. The fit reference profile is overlaid when available.

In [ ]:
def finite_sld_profiles(prediction):
    sld_x = to_numpy(prediction["sampled_sld_xaxis"])
    profiles = to_numpy(prediction["sampled_sld_profiles"])
    loglike = to_numpy(prediction["log_likelihoods"]).ravel()
    if profiles.ndim == 1:
        profiles = profiles[None, :]
    mask = np.isfinite(loglike) & np.all(np.isfinite(profiles), axis=1)
    profiles = profiles[mask]
    loglike = loglike[mask]
    if sld_x.ndim == 2:
        sld_x = sld_x[mask]
    if len(profiles) == 0:
        raise ValueError("No finite sampled SLD profiles available to plot.")
    return sld_x, profiles, loglike


from math import erf, sqrt


def smooth_reference_sld_profile(name, x):
    ref = FIT_REFERENCE.get(name)
    if ref is None:
        return None
    x = np.asarray(x, dtype=float)
    interfaces = np.array([0.0, ref["Pt thickness"], ref["Pt thickness"] + ref["Co thickness"]])
    slds = np.array([0.0, ref["Pt SLD"], ref["effective Co SLD"], ref["Al2O3 SLD"]])
    roughnesses = np.array([ref["air/Pt roughness"], ref["Pt/Co roughness"], ref["Co/Al2O3 roughness"]])
    y = np.full_like(x, slds[0], dtype=float)
    for z0, sigma, left, right in zip(interfaces, roughnesses, slds[:-1], slds[1:]):
        if sigma > 0:
            transition = 0.5 * (1.0 + np.vectorize(erf)((x - z0) / (sqrt(2.0) * sigma)))
        else:
            transition = (x >= z0).astype(float)
        y += (right - left) * transition
    return y


def plot_sampled_sld_profiles(ax, name, prediction):
    sld_x, profiles, loglike = finite_sld_profiles(prediction)
    best = int(np.argmax(loglike))
    order = np.argsort(loglike)
    line_alpha = min(0.18, max(0.01, 30 / len(order)))
    for idx in order:
        x = sld_x[idx] if np.ndim(sld_x) == 2 else sld_x
        ax.plot(x, profiles[idx], color="0.25", alpha=line_alpha, lw=0.8)
    x_best = sld_x[best] if np.ndim(sld_x) == 2 else sld_x
    ax.plot(x_best, profiles[best], color="crimson", lw=2.4, label="MAP")
    reference = smooth_reference_sld_profile(name, x_best)
    if reference is not None:
        ax.plot(x_best, reference, color="royalblue", ls="--", lw=2.2, label="fit reference")
    ax.set(xlabel="Depth (A)", ylabel="SLD (1e-6 A^-2)", title=f"{name} sampled SLD profiles")
    ax.legend()
    return len(profiles)


fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 4.5), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, (name, result) in zip(axes, results.items()):
    count = plot_sampled_sld_profiles(ax, name, result["prediction"])
    print(f"{name}: plotted {count} finite sampled SLD profiles")
plt.show()
